In [5]:
from pathlib import Path
import os, sys

# Determine repository root in a portable way.
cwd = Path.cwd()
if cwd.name == "src":
    repo_root = cwd.parent
else:
    repo_root = cwd

if not (repo_root / "src" / "agent_chatbot").exists():
    p = repo_root
    while p != p.parent and not (p / "src" / "agent_chatbot").exists():
        p = p.parent
    repo_root = p

print("repo_root:", repo_root)
agent_chatbot_path = repo_root / "src" / "agent_chatbot"

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(agent_chatbot_path))

from agent_chatbot.LeChatBot import (
    create_document_from_json,
    create_index_from_documents,
    run_chatbot,
    Settings,
)

from agent_chatbot.agent_tools import get_player_stats_tool, get_team_stats_tool
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool, ToolMetadata

repo_root: /Users/josephthi/Documents/Personal Projects/AI Basketball Predictor/basketball-ai


In [6]:
json_path = agent_chatbot_path / "nba_wikipedia_corpus.json"
print("json_path exists:", json_path.exists())
if not json_path.exists():
    raise FileNotFoundError(f"{json_path} not found; check repository contents")

documents = create_document_from_json(str(json_path))
document_index = create_index_from_documents(documents)
query_engine = document_index.as_query_engine(similarity_top_k=2)

storage_dir = agent_chatbot_path / "storage"
if not storage_dir.exists():
    document_index.storage_context.persist(persist_dir=str(storage_dir))
else:
    storage_context = StorageContext.from_defaults(persist_dir=str(storage_dir))
    document_index = load_index_from_storage(storage_context)

rag_tool = QueryEngineTool(
    query_engine=query_engine,
    metadata=ToolMetadata(
        name="nba_history_tool",
        description="RAG general NBA history + narrative search."
    )
)
player_stats_tool = get_player_stats_tool()
team_stats_tool = get_team_stats_tool()

agent = ReActAgent(
    tools=[rag_tool, player_stats_tool, team_stats_tool],
    llm=Settings.llm,
    verbose=True,
)

json_path exists: True


# Ollama Download Link
https://ollama.com/download/mac


After download you're going to want to:
* ollama serve (starts the server)
* ollama pull llama3 (gets the llama3 model)

In [9]:
"""
In order to run this you will need to run a local ollama
model using ollama serve, after it runs a window should
pop up where you could type in questions about NBA players

Type exit to quit the chatbot.
"""

!ollama serve #used to activate ollama server in notebook
import asyncio
await asyncio.to_thread(lambda: asyncio.run(run_chatbot(agent)))  # you can run in notebook kernel

Error: listen tcp 127.0.0.1:11434: bind: address already in use
NBA Analyst Agent is ready! (Type 'exit' to quit)

Response: Stephen Curry is a legendary NBA player known for his incredible shooting ability and clutch performances. Throughout his career, he has averaged around 22 points per game, with a high of 30.1 in the 2015-16 season. He has also been an elite three-point shooter, averaging over 4 makes per game. Curry has led the Golden State Warriors to multiple championships and has been named NBA MVP twice.

Response: LeBron James is an incredibly accomplished basketball player. He has achieved numerous milestones throughout his career, including five Finals triple-doubles and becoming the youngest player to reach 27,000 career points.

Response: Based on the provided context information, there isn't enough data to determine the best power forward. However, if we were to analyze the 2011-12 NBA season, we could look at players like Antawn Jamison (Cleveland Cavaliers), Dirk Now